# Build: public datasets for the wider complement / C4b exploration

Downloads are done outside the notebook (GEO supplementary files; see `scripts/download_public_datasets.sh`). This notebook turns the downloaded files into harmonised `.h5ad` objects using the loaders in `scripts/oligoc4b_public.py`.

| Dataset | Accession | Species / modality | Comparison | Why |
| --- | --- | --- | --- | --- |
| Park et al. 2023 AD hippocampus | GSE224398 | mouse scRNA-seq | App^NL-G-F^ vs control, 1/3/6 mo | the dataset in which the C4b⁺ DA-oligodendrocyte state was described |
| Aging snRNA-seq HIP + CP | GSE212576 | mouse snRNA-seq | old vs young | aging counterpart already used in this repo |
| Ximerakis et al. 2019 | GSE129788 | mouse scRNA-seq, whole brain | old vs young | independent aging dataset with author cell types |
| Kaya et al. 2022 | GSE202579 | mouse scRNA-seq, white vs grey matter | aged vs young | interferon-responsive oligodendrocytes in aging white matter |
| Zhou et al. 2020 | GSE140511 | mouse snRNA-seq, cortex + hippocampus | 5XFAD vs non-Tg (± Trem2-KO) | second AD model |
| Chen et al. 2020 | GSE152506 | mouse Spatial Transcriptomics | App^NL-G-F^ vs WT, 3–18 mo | whole-transcriptome spatial AD data; C4b is a plaque-induced gene |
| Jäkel et al. 2019 | GSE118257 | human snRNA-seq, MS white matter | MS lesions vs control | human MS oligodendrocytes |
| Absinta et al. 2021 | GSE180759 | human snRNA-seq, chronic active MS lesions | lesion edge / core / periplaque vs control | human MS lesion rim biology |

Harmonised `obs` columns: `dataset`, `species`, `modality`, `sample`, `group` (two-level comparison), `group_ref`, `cell_type_coarse`, `cell_type_original`. Coarse cell types come from author annotations when provided, otherwise from marker-based cluster annotation (`annotate_by_markers`).

Environment: `OLIGOC4B_PUBLIC_RAW_DIR` (downloaded files, one folder per accession) and `OLIGOC4B_PUBLIC_PROCESSED_DIR` (output).

In [1]:
import os, sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../../scripts")
import importlib, oligoc4b_public as pub
importlib.reload(pub)
import pandas as pd
print("raw:", pub.RAW_DIR); print("processed:", pub.PROCESSED_DIR)
print("available loaders:", list(pub.LOADERS))

raw: /Volumes/jamboree/OligoC4b_public/raw
processed: /Volumes/jamboree/OligoC4b_public/processed
available loaders: ['Park2023_AD_hippocampus_scRNA', 'Aging_snRNA_HIP_CP_mouse', 'Ximerakis2019_aging_brain_scRNA', 'Zhou2020_5XFAD_snRNA', 'Jakel2019_MS_human_snRNA', 'Absinta2021_MS_human_snRNA', 'Chen2020_ST_AppNLGF_mouse', 'Kaya2022_aged_WM_vs_GM_scRNA']


In [2]:
built = pub.build_all(overwrite=False)
built

[skip] Park2023_AD_hippocampus_scRNA exists
[skip] Aging_snRNA_HIP_CP_mouse exists
[skip] Ximerakis2019_aging_brain_scRNA exists
[skip] Zhou2020_5XFAD_snRNA exists
[skip] Jakel2019_MS_human_snRNA exists
[skip] Absinta2021_MS_human_snRNA exists
[skip] Chen2020_ST_AppNLGF_mouse exists
[skip] Kaya2022_aged_WM_vs_GM_scRNA exists


{'Park2023_AD_hippocampus_scRNA': '/Volumes/jamboree/OligoC4b_public/processed/Park2023_AD_hippocampus_scRNA.h5ad',
 'Aging_snRNA_HIP_CP_mouse': '/Volumes/jamboree/OligoC4b_public/processed/Aging_snRNA_HIP_CP_mouse.h5ad',
 'Ximerakis2019_aging_brain_scRNA': '/Volumes/jamboree/OligoC4b_public/processed/Ximerakis2019_aging_brain_scRNA.h5ad',
 'Zhou2020_5XFAD_snRNA': '/Volumes/jamboree/OligoC4b_public/processed/Zhou2020_5XFAD_snRNA.h5ad',
 'Jakel2019_MS_human_snRNA': '/Volumes/jamboree/OligoC4b_public/processed/Jakel2019_MS_human_snRNA.h5ad',
 'Absinta2021_MS_human_snRNA': '/Volumes/jamboree/OligoC4b_public/processed/Absinta2021_MS_human_snRNA.h5ad',
 'Chen2020_ST_AppNLGF_mouse': '/Volumes/jamboree/OligoC4b_public/processed/Chen2020_ST_AppNLGF_mouse.h5ad',
 'Kaya2022_aged_WM_vs_GM_scRNA': '/Volumes/jamboree/OligoC4b_public/processed/Kaya2022_aged_WM_vs_GM_scRNA.h5ad'}

In [3]:
import scanpy as sc
rows = []
for name, path in pub.processed_paths().items():
    a = sc.read_h5ad(path, backed="r")
    o = a.obs
    rows.append({"dataset": name, "n_cells": a.n_obs, "n_genes": a.n_vars, "species": o["species"].iloc[0], "modality": o["modality"].iloc[0],
                 "n_samples": o["sample"].nunique(), "groups": o["group"].value_counts().to_dict(),
                 "oligodendrocytes": int((o["cell_type_coarse"].astype(str) == "Oligodendrocyte").sum()),
                 "microglia": int((o["cell_type_coarse"].astype(str) == "Microglia").sum())})
summary = pd.DataFrame(rows).set_index("dataset")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 80)
display(summary)

,n_cells,n_genes,species,modality,n_samples,groups,oligodendrocytes,microglia
dataset,,,,,,,,
Absinta2021_MS_human_snRNA,66432,27807,human,snRNA,8,"{'MS': 61141, 'Ctrl': 5291}",45697,5432
Aging_snRNA_HIP_CP_mouse,109822,27213,mouse,snRNA,8,"{'Young': 60384, 'Old': 49438}",19593,3112
Chen2020_ST_AppNLGF_mouse,19021,44669,mouse,spatial,20,"{'WT': 9542, 'AD': 9479}",0,0
Jakel2019_MS_human_snRNA,17790,21349,human,snRNA,20,"{'MS': 11199, 'Ctrl': 6591}",7973,428
Kaya2022_aged_WM_vs_GM_scRNA,29623,20027,mouse,scRNA,8,"{'WM': 23391, 'GM': 6232}",15776,5054
Park2023_AD_hippocampus_scRNA,47660,21809,mouse,scRNA,6,"{'AD': 27003, 'WT': 20657}",10237,13952
Ximerakis2019_aging_brain_scRNA,37069,14698,mouse,scRNA,16,"{'Old': 21041, 'Young': 16028}",12384,3910
Zhou2020_5XFAD_snRNA,125858,24693,mouse,snRNA,20,"{'WT': 69841, 'AD': 56017}",17356,4152


In [4]:
# sanity check of the coarse annotation: cell-type composition per dataset
for name, path in pub.processed_paths().items():
    a = sc.read_h5ad(path, backed="r")
    ct = a.obs["cell_type_coarse"].astype(str).value_counts()
    print(f"\n{name}: ", ct.to_dict())
    if "cell_type_original" in a.obs:
        cross = pd.crosstab(a.obs["cell_type_original"].astype(str), a.obs["cell_type_coarse"].astype(str))
        # show which original labels ended up where (top 25 labels by size)
        display(cross.loc[cross.sum(axis=1).sort_values(ascending=False).index[:25]])


Absinta2021_MS_human_snRNA:  {'Oligodendrocyte': 45697, 'Astrocyte': 8211, 'Microglia': 5432, 'Neuron': 2803, 'OPC': 2175, 'Vascular/Fibroblast': 1708, 'Immune (lymphoid/myeloid)': 406}


cell_type_coarse,Astrocyte,Immune (lymphoid/myeloid),Microglia,Neuron,OPC,Oligodendrocyte,Vascular/Fibroblast
cell_type_original,,,,,,,
oligodendrocytes,0,0,0,0,0,45697,0
astrocytes,8211,0,0,0,0,0,0
immune,0,0,5432,0,0,0,0
neurons,0,0,0,2803,0,0,0
opc,0,0,0,0,2175,0,0
vascular_cells,0,0,0,0,0,0,1708
lymphocytes,0,406,0,0,0,0,0



Aging_snRNA_HIP_CP_mouse:  {'Neuron': 67036, 'Oligodendrocyte': 19593, 'Astrocyte': 12690, 'OPC': 4754, 'Microglia': 3112, 'Vascular/Fibroblast': 1065, 'Endothelial': 1012, 'Immune (lymphoid/myeloid)': 350, 'Ependymal': 210}



Chen2020_ST_AppNLGF_mouse:  {'spot': 19021}

Jakel2019_MS_human_snRNA:  {'Oligodendrocyte': 7973, 'Neuron': 5031, 'Astrocyte': 1242, 'Endothelial': 836, 'Immune (lymphoid/myeloid)': 782, 'Vascular/Fibroblast': 697, 'Other': 449, 'Microglia': 428, 'OPC': 352}


cell_type_coarse,Astrocyte,Endothelial,Immune (lymphoid/myeloid),Microglia,Neuron,OPC,Oligodendrocyte,Other,Vascular/Fibroblast
cell_type_original,,,,,,,,,
Oligo2,0,0,0,0,0,0,1839,0,0
Oligo4,0,0,0,0,0,0,1579,0,0
Neuron1,0,0,0,0,1507,0,0,0,0
Oligo6,0,0,0,0,0,0,1484,0,0
Neuron2,0,0,0,0,1438,0,0,0,0
Oligo5,0,0,0,0,0,0,1167,0,0
Oligo1,0,0,0,0,0,0,1129,0,0
Astrocytes,1046,0,0,0,0,0,0,0,0
Neuron4,0,0,0,0,948,0,0,0,0



Kaya2022_aged_WM_vs_GM_scRNA:  {'Oligodendrocyte': 15776, 'Immune (lymphoid/myeloid)': 5877, 'Microglia': 5054, 'Astrocyte': 1933, 'Neuron': 546, 'Endothelial': 181, 'Ependymal': 166, 'OPC': 90}



Park2023_AD_hippocampus_scRNA:  {'Microglia': 13952, 'Oligodendrocyte': 10237, 'Astrocyte': 7470, 'Endothelial': 5107, 'Neuron': 3298, 'OPC': 2471, 'Ependymal': 1906, 'Immune (lymphoid/myeloid)': 1644, 'Vascular/Fibroblast': 1575}

Ximerakis2019_aging_brain_scRNA:  {'Oligodendrocyte': 12384, 'Astrocyte': 6747, 'Other': 5301, 'Microglia': 3910, 'Endothelial': 2413, 'OPC': 2187, 'Vascular/Fibroblast': 1655, 'Neuron': 1368, 'Immune (lymphoid/myeloid)': 700, 'Ependymal': 404}


cell_type_coarse,Astrocyte,Endothelial,Ependymal,Immune (lymphoid/myeloid),Microglia,Neuron,OPC,Oligodendrocyte,Other,Vascular/Fibroblast
cell_type_original,,,,,,,,,,
OLG,0,0,0,0,0,0,0,12384,0,0
ASC,6747,0,0,0,0,0,0,0,0,0
mNEUR,0,0,0,0,0,0,0,0,5135,0
MG,0,0,0,0,3910,0,0,0,0,0
EC,0,2413,0,0,0,0,0,0,0,0
OPC,0,0,0,0,0,0,2187,0,0,0
OEG,0,0,0,0,0,892,0,0,0,0
PC,0,0,0,0,0,0,0,0,0,735
NendC,0,0,0,0,0,394,0,0,0,0



Zhou2020_5XFAD_snRNA:  {'Neuron': 91680, 'Oligodendrocyte': 17356, 'Astrocyte': 7217, 'Microglia': 4152, 'OPC': 3261, 'Vascular/Fibroblast': 2192}
